# Normalisasi Slang, Ekstraksi Bigram & Trigram

**Tujuan**: Melakukan normalisasi kata slang/gaul ke bentuk baku, kemudian ekstraksi bigram dan trigram untuk memperkaya fitur teks sebelum topic modeling.

**Langkah-langkah:**
1. Load data hasil preprocessing
2. Normalisasi slang/abbreviation ke bentuk baku
3. Ekstraksi bigram menggunakan Gensim Phrases
4. Ekstraksi trigram berdasarkan bigram
5. Visualisasi dan analisis
6. Simpan hasil ke normalisasi_slang.csv

## 1. Setup dan Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import ast
import os
import warnings
warnings.filterwarnings('ignore')

from gensim.models import Phrases
from gensim.models.phrases import Phraser

print("Library berhasil diimpor!")

## 2. Load Data

In [ ]:
df = pd.read_csv('data/hasil_processing.csv')
print(f"Shape: {df.shape}")
print(f"\nKolom: {df.columns.tolist()}")

def parse_tokens(val):
    if pd.isna(val) or str(val).strip() == '' or str(val) == '[]':
        return []
    try:
        tokens = ast.literal_eval(str(val))
        if isinstance(tokens, list):
            return [str(t) for t in tokens if str(t).strip() != '']
        return []
    except:
        return []

df['tokens_parsed'] = df['tokens'].apply(parse_tokens)

df_valid = df[df['tokens_parsed'].apply(len) > 0].copy()
print(f"\nJumlah baris dengan tokens: {len(df_valid)}")
print(f"Jumlah baris tanpa tokens (dilewati): {len(df) - len(df_valid)}")

## 3. Normalisasi Slang

In [ ]:
slang_dict = {
    'yg': 'yang', 'gk': 'tidak', 'gak': 'tidak', 'nggak': 'tidak', 'nggk': 'tidak',
    'ga': 'tidak', 'klo': 'kalau', 'klau': 'kalau', 'kalo': 'kalau',
    'dg': 'dengan', 'dgn': 'dengan', 'krn': 'karena', 'krna': 'karena', 'karna': 'karena',
    'utk': 'untuk', 'jd': 'jadi', 'jdi': 'jadi', 'bs': 'bisa', 'dr': 'dari',
    'dri': 'dari', 'sblm': 'sebelum', 'stlh': 'setelah',
    'udah': 'sudah', 'sdh': 'sudah', 'blm': 'belum',
    'tp': 'tapi', 'tpi': 'tetapi', 'trs': 'terus',
    'lg': 'lagi', 'sdg': 'sedang', 'sdgkn': 'sedangkan',
    'aja': 'saja', 'banget': 'sangat', 'bgt': 'sangat', 'bnget': 'sangat',
    'gitu': 'begitu', 'gt': 'begitu', 'gimana': 'bagaimana', 'gmna': 'bagaimana',
    'gmn': 'bagaimana', 'kpn': 'kapan', 'knp': 'mengapa', 'knpa': 'mengapa',
    'gpp': 'tidak apa-apa', 'gapapa': 'tidak apa-apa',
    'ny': 'nya', 'nye': 'nya', 'tu': 'itu', 'nih': 'ini', 'ni': 'ini',
    'deh': 'saja', 'dong': 'saja', 'sih': 'saja', 'kok': 'kenapa',
    'aku': 'saya', 'gue': 'saya', 'gw': 'saya', 'gua': 'saya', 'aq': 'saya',
    'kamu': 'anda', 'elu': 'anda', 'lu': 'anda', 'lo': 'anda',
    'sy': 'saya', 'mu': 'anda',
    'pak': 'bapak', 'bu': 'ibu', 'kak': 'kakak', 'bang': 'abang',
    'mas': 'mas', 'mbak': 'mbak', 'teh': 'kakak',
    'mau': 'akan', 'mo': 'akan', 'pengen': 'ingin', 'pingin': 'ingin', 'pengin': 'ingin',
    'liat': 'lihat', 'denger': 'dengar', 'ngomong': 'bicara',
    'bikin': 'buat', 'ckp': 'cukup',
    'pinter': 'pintar', 'deket': 'dekat',
    'gini': 'begini', 'bgitu': 'begitu',
    'spt': 'seperti', 'sperti': 'seperti', 'macem': 'macam', 'cem': 'macam',
    'ama': 'dengan', 'sm': 'dengan', 'sama': 'dengan',
    'soalnya': 'karena',
    'msh': 'masih', 'masi': 'masih',
    'cm': 'cuma', 'cmn': 'cuma', 'cuman': 'cuma',
    'jg': 'juga', 'ogah': 'tidak mau',
    'belom': 'belum', 'blom': 'belum',
    'udh': 'sudah', 'udha': 'sudah',
    'prnh': 'pernah',
    'tlng': 'tolong', 'pls': 'tolong', 'please': 'tolong',
    'thx': 'terima kasih', 'makasih': 'terima kasih', 'trims': 'terima kasih',
    'thanks': 'terima kasih', 'thank': 'terima kasih',
    'sorry': 'maaf', 'maap': 'maaf', 'sori': 'maaf',
    'yes': 'ya', 'ok': 'oke', 'sip': 'siap', 'siapp': 'siap',
    'yuk': 'mari', 'ayo': 'mari',
    'wes': 'sudah', 'wis': 'sudah',
    'g': 'tidak', 'gg': 'tidak',
    'spy': 'supaya', 'biar': 'supaya', 'biarin': 'biarkan',
    'musti': 'harus', 'kudu': 'harus', 'hrs': 'harus',
    'gaada': 'tidak ada', 'gada': 'tidak ada', 'gakada': 'tidak ada',
    'gabisa': 'tidak bisa', 'gbs': 'tidak bisa',
    'gatau': 'tidak tahu', 'gtau': 'tidak tahu',
    'sp': 'siapa', 'apaan': 'apa',
    'y': 'ya', 'iya': 'ya', 'iy': 'ya',
    'ndak': 'tidak', 'bkn': 'bukan',
    'jgn': 'jangan', 'jngn': 'jangan',
    'no': 'tidak', 'nope': 'tidak',
    'skali': 'sangat',
    'lbh': 'lebih', 'plg': 'paling', 'krg': 'kurang',
    'pdhl': 'padahal',
    'wlkp': 'walaupun', 'mskp': 'meskipun',
    'smp': 'sampai', 'sampe': 'sampai', 'sampek': 'sampai',
    'shg': 'sehingga',
    'cth': 'contoh',
    'dll': 'dan lain-lain', 'dsb': 'dan sebagainya', 'dst': 'dan seterusnya',
    'smua': 'semua',
    'brp': 'berapa', 'bnyk': 'banyak', 'sdkt': 'sedikit',
    'rata2': 'rata-rata',
    'kdng': 'kadang', 'kadang2': 'kadang-kadang',
    'hmpir': 'hampir',
    'bener': 'benar', 'bnr': 'benar',
    'seneng': 'senang',
    'nangis': 'menangis',
    'ketawa': 'tertawa', 'ngakak': 'tertawa',
    'bete': 'kesal', 'bt': 'bosan', 'bosen': 'bosan',
    'galau': 'gelisah', 'kuatir': 'khawatir',
    'pusing': 'pusing', 'bingung': 'bingung',
    'bimbang': 'bingung',
    'relax': 'tenang', 'calm': 'tenang', 'santai': 'tenang',
    'safe': 'aman',
    'fail': 'gagal', 'broken': 'rusak',
    'nice': 'baik', 'cool': 'keren', 'good': 'baik', 'fine': 'baik',
    'jos': 'baik', 'mantul': 'baik', 'top': 'baik',
    'bgs': 'baik', 'jlk': 'buruk',
    'parah': 'buruk', 'busuk': 'buruk',
    'gila': 'gila', 'edann': 'gila', 'edan': 'gila', 'gile': 'gila', 'gilak': 'gila',
    'n': 'dan',
    'd': 'di',
    'w': 'saya',
    'kalian': 'anda',
    'training': 'pelatihan',
    'klb': 'kejadian luar biasa',
    'asn': 'aparatur sipil negara', 'cpns': 'calon pegawai negeri sipil',
    'pppk': 'pegawai pemerintah dengan perjanjian kerja',
    'rapel': 'rekap', 'ordal': 'orang dalam',
    'mjd': 'menjadi', 'mhs': 'mahasiswa',
    'gimna': 'bagaimana',
    'kl': 'kalau',
    'kaga': 'tidak', 'kagak': 'tidak',
    'tuh': 'itu', 'toh': 'itu',
    'lah': 'lah', 'pun': 'pun', 'loh': 'loh', 'lho': 'lho',
    'bgn': 'badan gizi nasional',
    'mbg': 'makan bergizi gratis',
    'ag': 'ahli gizi',
    'sppg': 'satuan pelayanan pangan gizi',
    'gizi': 'gizi',
    'dokter': 'dokter',
    'pt': 'perusahaan',
    'sop': 'standar operasional prosedur',
    'rs': 'rumah sakit',
    'str': 'surat tanda registrasi',
    'qc': 'quality control',
    'uu': 'undang-undang',
    'yth': 'yang terhormat',
    'lok': 'lowongan', 'loker': 'lowongan kerja',
    'bngt': 'banget',
    'pake': 'pakai',
    'make': 'pakai',
    'nya': 'nya',
    'yang': 'yang',
    'dengan': 'dengan',
    'karena': 'karena',
    'untuk': 'untuk',
    'dari': 'dari',
    'sudah': 'sudah',
    'belum': 'belum',
    'jadi': 'jadi',
    'bisa': 'bisa',
    'tapi': 'tetapi',
    'kalau': 'kalau',
    'tidak': 'tidak',
    'saja': 'saja',
    'sangat': 'sangat',
    'begitu': 'begitu',
    'bagaimana': 'bagaimana',
    'tidak apa-apa': 'tidak apa-apa',
    'cuma': 'cuma',
    'masih': 'masih',
    'lagi': 'lagi',
    'juga': 'juga',
    'sedang': 'sedang',
    'sedangkan': 'sedangkan',
    'tetapi': 'tetapi',
    'terus': 'terus',
    'saya': 'saya',
    'anda': 'anda',
    'ini': 'ini',
    'itu': 'itu',
    'nya': 'nya',
    'ku': 'saya',
    'di': 'di',
    'ke': 'ke',
    'dan': 'dan',
    'ada': 'ada',
    'bukan': 'bukan',
    'akan': 'akan',
    'harus': 'harus',
    'ingin': 'ingin',
    'mau': 'mau',
    'pada': 'pada',
    'oleh': 'oleh',
    'jika': 'jika',
    'atau': 'atau',
    'dalam': 'dalam',
    'adalah': 'adalah',
    'bahwa': 'bahwa',
    'sebagai': 'sebagai',
    'seperti': 'seperti',
    'lebih': 'lebih',
    'kurang': 'kurang',
    'paling': 'paling',
    'cukup': 'cukup',
    'hanya': 'hanya',
    'semua': 'semua',
    'setiap': 'setiap',
    'banyak': 'banyak',
    'sedikit': 'sedikit',
    'beberapa': 'beberapa',
    'sering': 'sering',
    'jarang': 'jarang',
    'selalu': 'selalu',
    'kadang': 'kadang',
    'pernah': 'pernah',
    'telah': 'sudah',
    'sebelum': 'sebelum',
    'setelah': 'setelah',
    'kemudian': 'kemudian',
    'lalu': 'lalu',
    'saat': 'saat',
    'ketika': 'ketika',
    'sementara': 'sementara',
    'sejak': 'sejak',
    'sampai': 'sampai',
    'hingga': 'hingga',
    'antara': 'antara',
    'yaitu': 'yaitu',
    'yakni': 'yakni',
    'misalnya': 'misalnya',
    'contoh': 'contoh',
    'termasuk': 'termasuk',
    'terdiri': 'terdiri',
    'berasal': 'berasal',
    'berubah': 'berubah',
    'menjadi': 'menjadi',
    'menurut': 'menurut',
    'berdasarkan': 'berdasarkan',
    'demi': 'demi',
    'guna': 'guna',
    'agar': 'agar',
    'supaya': 'supaya',
    'biar': 'supaya',
    'sehingga': 'sehingga',
    'maka': 'maka',
    'karenanya': 'karenanya',
    'oleh karena itu': 'oleh karena itu',
    'dengan demikian': 'dengan demikian',
    'oleh sebab itu': 'oleh sebab itu',
    'akibatnya': 'akibatnya',
    'walaupun': 'walaupun',
    'meskipun': 'meskipun',
    'biarpun': 'walaupun',
    'walau': 'walaupun',
    'sekalipun': 'walaupun',
    'padahal': 'padahal',
    'namun': 'namun',
    'melainkan': 'melainkan',
    'kecuali': 'kecuali',
    'selain': 'selain',
    'malah': 'malah',
    'malahan': 'malah',
    'justru': 'justru',
    'bahkan': 'bahkan',
    'apalagi': 'apalagi',
    'terutama': 'terutama',
    'khususnya': 'khususnya',
    'umumnya': 'umumnya',
    'pada umumnya': 'pada umumnya',
    'kebanyakan': 'kebanyakan',
    'sebagian': 'sebagian',
    'seluruh': 'seluruh',
    'tiap': 'setiap',
    'masing-masing': 'masing-masing',
    'berbagai': 'berbagai',
    'macam': 'macam',
    'jenis': 'jenis',
    'macam-macam': 'macam-macam',
    'lain': 'lain',
    'lainnya': 'lainnya',
    'yang lain': 'yang lain',
    'berbeda': 'berbeda',
    'serupa': 'serupa',
    'mirip': 'mirip',
    'laksana': 'seperti',
    'bak': 'seperti',
    'bagai': 'seperti',
    'ibarat': 'ibarat',
    'bagaikan': 'seperti',
    'umpama': 'seperti',
    'seolah-olah': 'seolah-olah',
    'seakan-akan': 'seakan-akan',
    'seolah': 'seolah-olah',
    'seakan': 'seakan-akan',
    'tentang': 'tentang',
    'mengenai': 'mengenai',
    'soal': 'soal',
    'berkaitan': 'berkaitan',
    'terkait': 'terkait',
    'berhubungan': 'berhubungan',
    'sesuai': 'sesuai',
    'bagi': 'bagi',
    'kepada': 'kepada',
    'terhadap': 'terhadap',
    'atas': 'atas',
    'menuju': 'menuju',
    'di dalam': 'di dalam',
    'di luar': 'di luar',
    'di atas': 'di atas',
    'di bawah': 'di bawah',
    'di depan': 'di depan',
    'di belakang': 'di belakang',
    'di samping': 'di samping',
    'di antara': 'di antara',
    'di tengah': 'di tengah',
    'sekitar': 'sekitar',
    'keliling': 'keliling',
    'sekeliling': 'sekeliling',
    'sepanjang': 'sepanjang',
    'melalui': 'melalui',
    'lewat': 'melalui',
    'via': 'melalui',
    'melintasi': 'melintasi',
    'menembus': 'menembus',
    'melewati': 'melewati'
}

print(f"Jumlah entri kamus slang: {len(slang_dict)}")
print(f"Contoh: 'gk' -> '{slang_dict['gk']}', 'gak' -> '{slang_dict['gak']}', 'yg' -> '{slang_dict['yg']}'")

In [ ]:
def normalize_tokens(tokens, slang_dict):
    normalized = []
    for token in tokens:
        if token in slang_dict:
            normalized.append(slang_dict[token])
        else:
            normalized.append(token)
    return normalized

df_valid['tokens_normalized'] = df_valid['tokens_parsed'].apply(lambda x: normalize_tokens(x, slang_dict))

print("Contoh normalisasi slang:")
print("-" * 80)
for i in range(5):
    original = df_valid['tokens_parsed'].iloc[i]
    normalized = df_valid['tokens_normalized'].iloc[i]
    if original != normalized:
        print(f"Sebelum: {' '.join(original[:20])}")
        print(f"Sesudah: {' '.join(normalized[:20])}")
        print("-" * 80)

normalized_all = [t for tokens in df_valid['tokens_normalized'] for t in tokens]
original_all = [t for tokens in df_valid['tokens_parsed'] for t in tokens]

changes = sum(1 for o, n in zip(original_all, normalized_all) if o != n)
print(f"\nTotal perubahan: {changes} dari {len(original_all)} token ({changes/len(original_all)*100:.1f}%)")

## 4. Ekstraksi Bigram

In [ ]:
print("=" * 60)
print("EKSTRAKSI BIGRAM")
print("=" * 60)

tokens_for_bigram = df_valid['tokens_normalized'].tolist()

bigram_model = Phrases(tokens_for_bigram, min_count=10, threshold=10, delimiter='_')
bigram = Phraser(bigram_model)

bigram_tokens = [bigram[doc] for doc in tokens_for_bigram]
df_valid['tokens_bigram'] = bigram_tokens

bigrams_found = []
for doc in bigram_tokens:
    for token in doc:
        if '_' in token:
            bigrams_found.append(token)

bigram_freq = Counter(bigrams_found)
top_20_bigrams = bigram_freq.most_common(20)

print(f"\nJumlah token dengan bigram: {len(bigram_tokens)}")
print(f"Jumlah bigram unik: {len(bigram_freq)}")
print(f"\nTop 20 Bigram:")
for bigram, count in top_20_bigrams:
    print(f"  {bigram}: {count}")

fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*top_20_bigrams)
sns.barplot(x=list(counts), y=list(words), palette='Purples_r', ax=ax)
ax.set_title('Top 20 Bigram Terbentuk', fontsize=14, fontweight='bold')
ax.set_xlabel('Frekuensi')
plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/top_bigrams.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Ekstraksi Trigram

In [ ]:
print("=" * 60)
print("EKSTRAKSI TRIGRAM")
print("=" * 60)

trigram_model = Phrases(bigram[tokens_for_bigram], min_count=5, threshold=10, delimiter='_')
trigram = Phraser(trigram_model)

trigram_tokens = [trigram[bigram[doc]] for doc in tokens_for_bigram]
df_valid['tokens_trigram'] = trigram_tokens

trigrams_found = []
for doc in trigram_tokens:
    for token in doc:
        if token.count('_') >= 2:
            trigrams_found.append(token)

trigram_freq = Counter(trigrams_found)
top_20_trigrams = trigram_freq.most_common(20)

print(f"\nJumlah token dengan trigram: {len(trigram_tokens)}")
print(f"Jumlah trigram unik: {len(trigram_freq)}")
print(f"\nTop 20 Trigram:")
for trigram, count in top_20_trigrams:
    print(f"  {trigram}: {count}")

fig, ax = plt.subplots(figsize=(12, 6))
if len(top_20_trigrams) > 0:
    words, counts = zip(*top_20_trigrams)
    sns.barplot(x=list(counts), y=list(words), palette='Oranges_r', ax=ax)
    ax.set_title('Top 20 Trigram Terbentuk', fontsize=14, fontweight='bold')
    ax.set_xlabel('Frekuensi')
    plt.tight_layout()
    plt.savefig('output/top_trigrams.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Tidak ada trigram yang terbentuk dengan threshold saat ini.")

## 6. Visualisasi Perbandingan

In [ ]:
df_valid['token_count_original'] = df_valid['tokens_parsed'].apply(len)
df_valid['token_count_normalized'] = df_valid['tokens_normalized'].apply(len)
df_valid['token_count_bigram'] = df_valid['tokens_bigram'].apply(len)
df_valid['token_count_trigram'] = df_valid['tokens_trigram'].apply(len)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

stages = [
    ('Original', 'token_count_original', 'Reds_r'),
    ('Normalized', 'token_count_normalized', 'Greens_r'),
    ('Bigram', 'token_count_bigram', 'Blues_r'),
    ('Trigram', 'token_count_trigram', 'Purples_r')
]

for idx, (title, col, palette) in enumerate(stages):
    axes[idx].hist(df_valid[col], bins=30, color='gray', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{title}\nMean: {df_valid[col].mean():.1f}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Jumlah Token')
    axes[idx].set_ylabel('Frekuensi')
    axes[idx].axvline(df_valid[col].mean(), color='red', linestyle='--', linewidth=2)

plt.suptitle('Distribusi Jumlah Token per Tahap', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/token_count_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRingkasan Jumlah Token:")
print(f"  Original:   Mean={df_valid['token_count_original'].mean():.2f}, Median={df_valid['token_count_original'].median():.2f}")
print(f"  Normalized: Mean={df_valid['token_count_normalized'].mean():.2f}, Median={df_valid['token_count_normalized'].median():.2f}")
print(f"  Bigram:     Mean={df_valid['token_count_bigram'].mean():.2f}, Median={df_valid['token_count_bigram'].median():.2f}")
print(f"  Trigram:    Mean={df_valid['token_count_trigram'].mean():.2f}, Median={df_valid['token_count_trigram'].median():.2f}")

print("\nContoh hasil ekstraksi Bigram & Trigram:")
print("-" * 80)
for i in range(3):
    print(f"\nOriginal:   {' '.join(df_valid['tokens_parsed'].iloc[i][:15])}")
    print(f"Bigram:     {' '.join(df_valid['tokens_bigram'].iloc[i][:15])}")
    print(f"Trigram:    {' '.join(df_valid['tokens_trigram'].iloc[i][:15])}")

## 7. WordCloud Hasil Normalisasi

In [ ]:
from wordcloud import WordCloud

text_normalized = ' '.join(normalized_all)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wordcloud1 = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='viridis',
    max_words=100,
    min_font_size=10
).generate(text_normalized)

axes[0].imshow(wordcloud1, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('WordCloud Setelah Normalisasi Slang', fontsize=14, fontweight='bold')

all_trigram_tokens = [t for tokens in trigram_tokens for t in tokens]
wordcloud2 = WordCloud(
    width=800, height=400,
    background_color='black',
    colormap='plasma',
    max_words=100,
    min_font_size=10
).generate(' '.join(all_trigram_tokens))

axes[1].imshow(wordcloud2, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('WordCloud Setelah Trigram', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('output/wordcloud_normalisasi.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Simpan Hasil

In [ ]:
df_output = df_valid.copy()

df_output['tokens_final'] = df_output['tokens_trigram']
df_output['text_final'] = df_output['tokens_trigram'].apply(lambda x: ' '.join(x))
df_output['token_count_final'] = df_output['tokens_trigram'].apply(len)

cols_to_keep = [
    'text', 'diggCount', 'replyCommentTotal', 'createTimeISO', 'uniqueId',
    'videoWebUrl', 'uid', 'cid', 'avatarThumbnail', 'source_file',
    'text_casefold', 'text_clean', 'tokens', 'token_count',
    'tokens_nostop', 'token_count_nostop',
    'tokens_parsed', 'tokens_normalized', 'tokens_bigram', 'tokens_trigram',
    'text_final', 'token_count_final'
]

available_cols = [c for c in cols_to_keep if c in df_output.columns]
df_output = df_output[available_cols]

os.makedirs('output', exist_ok=True)
df_output.to_csv('output/normalisasi_slang.csv', index=False)

print(f"Dataset disimpan ke: output/normalisasi_slang.csv")
print(f"Shape: {df_output.shape}")
print(f"\nKolom: {df_output.columns.tolist()}")
print("\nPreview hasil:")
df_output[['text', 'text_final', 'token_count_original', 'token_count_final']].head()

## 9. Summary

In [ ]:
print("=" * 60)
print("NORMALISASI SLANG & EKSTRAKSI N-GRAM COMPLETE!")
print("=" * 60)

print(f"\nTotal dokumen diproses: {len(df_valid):,}")
print(f"Bigram unik terbentuk: {len(bigram_freq):,}")
print(f"Trigram unik terbentuk: {len(trigram_freq):,}")

print(f"\nBigram paling sering:")
for bg, count in bigram_freq.most_common(10):
    print(f"  {bg}: {count}")

if len(trigram_freq) > 0:
    print(f"\nTrigram paling sering:")
    for tg, count in trigram_freq.most_common(10):
        print(f"  {tg}: {count}")

print(f"\nVisualisasi disimpan di folder output/")
print(f"  - output/top_bigrams.png")
print(f"  - output/top_trigrams.png")
print(f"  - output/token_count_comparison.png")
print(f"  - output/wordcloud_normalisasi.png")

print(f"\nDataset disimpan di: output/normalisasi_slang.csv")
print(f"\nLangkah berikutnya: Topic Modeling (LDA/NMF)")